## Moving The Drone

Open a terminal and run:
```bash
python3 scripts/drone-movement.py
```

In [ ]:
import argparse
from isaaclab.app import AppLauncher

parser = argparse.ArgumentParser(description="Fly the Drone")
AppLauncher.add_app_launcher_args(parser)
args_cli = parser.parse_args()

app_launcher = AppLauncher(args_cli)
simulation_app = app_launcher.app

import torch
import isaaclab.sim as sim_utils
from isaaclab.assets import Articulation
from isaaclab.sim import SimulationCfg, SimulationContext
from isaaclab_assets import CRAZYFLIE_CFG

def create_scene():

    # The Ground plane
    ground_config = sim_utils.GroundPlaneCfg()
    ground_config.func("/World/ground", ground_config)

    # The Lignt
    light_config = sim_utils.DomeLightCfg(intensity=3000.0)
    light_config.func("/World/Light", light_config)

def drone_object():

    # The Drone
    drone_config = CRAZYFLIE_CFG.replace(prim_path="/World/Crazyflie")
    drone_config.init_state.pos = (0.0, 0.0, 0.5)
    drone = Articulation(drone_config)
    return drone

def main():

    # The Simulation
    simulation_config = SimulationCfg(dt=0.01)
    simulation = SimulationContext(simulation_config)
    simulation.set_camera_view(eye=[0.6, 0.6, 0.9], target=[0.0, 0.0, 0.8])

    create_scene()
    drone = drone_object()
    simulation.reset()
    print("Simulation Started")

    body_id = drone.find_bodies("body")[0]
    robot_mass = drone.root_physx_view.get_masses().sum().item()
    print(f"Mass of the drone with id {body_id}:{robot_mass}")

    # g = 9.81
    robot_weight = robot_mass * 9.81

    forces = torch.zeros(1, 1, 3, device=drone.device)
    torques = torch.zeros(1, 1, 3, device=drone.device)

    target_z = 1.0
    kp = 6.0 # Proportional Gain
    kd = 3.0 # Derivative Gain

    count = 0
    while simulation_app.is_running():

        # Observation
        position = drone.data.root_pos_w[0]
        velocity = drone.data.root_lin_vel_w[0]

        # Decision
        height_error = target_z - position[2]
        thrust_z = robot_weight + robot_mass * (kp * height_error - kd * velocity[2])
        thrust_z = max(thrust_z, 0.0)
        forces[0, 0, 2] = thrust_z

        # Action
        drone.set_external_force_and_torque(forces, torques, body_ids=body_id)
        drone.write_data_to_sim()
        simulation.step()
        drone.update(simulation_config.dt)

        if count % 50 == 0:
            print(f"step {count:4d}, pos z = {position[2]:.3f} m, thrust = {thrust_z:.3f} N")
        count += 1

if __name__=="__main__":
    main()
    simulation_app.close()

### The Physics behind it:
We must maintain an upward thrust in order to keep the drone floating. This means that at a certain height forces on the drone must be equal, in order to achieve equilibrium. This formula can be derived.

However, we don't want the drone to just stay stably at a starting height; in fact, our aim is the achieve a constant altitude starting from the ground. Due to the reasons mentioned above, we can't achieve this with a constant thrust; therefore, we have to calculate an instantaneous force using a formula. In Isaac-Lab, the drone controlling system is **PD**(Proportional Derivative): This system modifies the thurust using how far the drone is from a target and how fast it is currently moving. The thrust that **PD Controller** commands is:

* $ thrust = W + m(k_p \times e-k_d\times v) $

Where $ W $ is the weight, $ m $ is the mass, $ e $ is the height error, $ v $ is the vertical velocity $ k_p $ is proportional gain, $ k_d $ is derivative gain. In order to have a better understanding of the principals behind a **PD controller** you can run the example application by using: 

```python
python -m http.server
```
**CAUTION:** If you are encountering an error related to the ports, you can try writing a port number (eg: python -m http.server 8080)
![Image](media/example_app.png)

-----------

The net upward force is equals to the weight subtracted from thrust pushing up. By using Newton's Second Law, we can find:

* $ m\times a = thrust - W $

Now, replacing $ thrust $ with the **PD** expression results in:

* $ m \times a = (W+m(kp\times e-kd\times v ) )-W  $

$ W $'s, weights, cancel out:

* $ m \times a = m(kp\times e-kd\times v ) $

$ m $'s, masses, cancel out:

* $ a = kp\times e-kd\times v $

This is a clear formulation showcases the primary purpose of the formula: Accelerate toward the target and, break against the motion. After that, in order to find how the error closes, we must calculate the error in terms of velocity and acceleration:

* $ e = target - z $

Target is a fixed number. We are taking the derivative of the $ e $. Moving up increases $ z $, which shrinks the error. Hence, we use minus sign for the $ \dot{z} $.

* $ \dot{e} = \frac{d}{dt} (target-z) = 0-\dot{z} = \dot{z} $

We can simplify this even further. In this equation \dot{z} corresponds to our vertical velocity:

* $ \dot{e} = −\dot{z} = -v $

Likewise, taking the derivative of $ \dot{z} $ will result in $ \ddot{z} $, which corresponds to vertical acceleration.

* $ \ddot{e} = −z̈ = -a $

By substituting the values we found into the place: 

* $ -\ddot{e} = kp\times e-kd\times −\dot{e} $

Moving one term to the other side gives us:

* $ kp\times e-kd\times −\dot{e} + \ddot{e} = 0 $

This equation proves that our drone behaves in a way which resambles damped harmonic oscillation. This type of motion has a universal template:

* $ \ddot{e} + 2\zeta \omega _n\times \dot{e} + \omega_{n}^{2}\times e = 0  $

Comparing our equation term by term with the template:

* $ \omega_{n}^{2}\times = k_p \Rightarrow \omega _n = \sqrt{k_p} $

* $ 2\zeta \omega _n = k_d \Rightarrow \zeta = \frac{k_d}{2\sqrt{k_p}} $

In our final formula, $ \omega _n $ is the natural frequency, while  $ \zeta $ is the damping ratio.

| Damping Ratio | Behaviour | Name |
| :---     | :----:     | ---:     |
| $ \zeta = 0 $ | Bounces forever | undamped |
| $ 0 < \zeta < 1 $ | Overshoots, then settles | underdamped |
| $ \zeta = 1 $ | Fastest settle with no overshoot | critically damped |
| $ \zeta > 1 $ | No overshoot, but slow | overdamped|

The tradeoff between underdamped and critically damped is speed versus control. Being critically damped ensures minimum bounciness, while being underdamped reaches the target fastest.
![Drone Gif](/media/drone_movement_new.gif)